# 10b. Optimización de Hiperparámetros de LightGBM (Alternativa B)

Este notebook corresponde a la **Fase 5: Tarea 5.1 - Optimización de Hiperparámetros (Tuning)** para la Alternativa B.
El objetivo es ajustar los hiperparámetros de LightGBM utilizando una estrategia de **búsqueda aleatoria (`RandomizedSearchCV`)** con una **Validación Cruzada Estratificada de 5 pliegues (Stratified 5-Fold CV)** en base a las matrices de la Alternativa B.

## 1. Carga de Librerías y Matrices Preprocesadas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import randint, uniform

from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "datos" / "processed"

X_TRAIN_PATH = DATA_DIR / "X_train_altB.csv"
X_TEST_PATH = DATA_DIR / "X_test_altB.csv"
Y_TRAIN_PATH = DATA_DIR / "y_train_altB.csv"
Y_TEST_PATH = DATA_DIR / "y_test_altB.csv"

print("Cargando matrices preprocesadas...")
X_train = pd.read_csv(X_TRAIN_PATH, sep=';', decimal=',')
X_test = pd.read_csv(X_TEST_PATH, sep=';', decimal=',')
y_train = pd.read_csv(Y_TRAIN_PATH, sep=';', decimal=',')['target']
y_test = pd.read_csv(Y_TEST_PATH, sep=';', decimal=',')['target']

print(f"\nDimensiones cargadas exitosamente:")
print(f"- X_train: {X_train.shape[0]} filas x {X_train.shape[1]} columnas")
print(f"- y_train: {y_train.shape[0]} registros")
print(f"- X_test : {X_test.shape[0]} filas x {X_test.shape[1]} columnas")
print(f"- y_test : {y_test.shape[0]} registros")

Cargando matrices preprocesadas...
- X_train: 7161 filas x 23 columnas
- y_train: 7161 registros
- X_test : 1791 filas x 23 columnas
- y_test : 1791 registros


## 2. Adecuación de Tipos de Datos

In [2]:
cols_bool = [col for col in X_train.columns if X_train[col].dtype == 'bool']
for col in cols_bool:
    X_train[col] = X_train[col].astype(int)
    X_test[col] = X_test[col].astype(int)

print(f"Columnas booleanas convertidas a entero: {len(cols_bool)}")

Columnas booleanas convertidas a entero: 7


## 3. Configuración y Ajuste del RandomizedSearchCV

In [3]:
param_dist = {
    'n_estimators': randint(100, 1000),
    'learning_rate': uniform(0.01, 0.19),
    'max_depth': randint(3, 13),
    'num_leaves': randint(20, 150),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_samples': randint(10, 50)
}

lgbm_base = LGBMClassifier(random_state=42, verbose=-1)
cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=lgbm_base,
    param_distributions=param_dist,
    n_iter=50,
    scoring='roc_auc',
    cv=cv_stratified,
    random_state=42,
    n_jobs=-1,
    verbose=0
)

print("Iniciando ajuste de RandomizedSearchCV (250 fits en total)...")
random_search.fit(X_train, y_train)
print("Ajuste de hiperparámetros finalizado")
print(f"\nMejor puntuación media de AUC-ROC en Validación Cruzada: {random_search.best_score_:.5f}")

Iniciando ajuste de RandomizedSearchCV (250 fits en total)...
¡Ajuste de hiperparámetros finalizado!
Mejor puntuación media de AUC-ROC en Validación Cruzada: 0.94377


## 4. Extracción y Redondeo Estricto de Hiperparámetros ("Diseño Humano")

In [4]:
best_params = {'colsample_bytree': 0.69, 'learning_rate': 0.06, 'max_depth': 9, 'min_child_samples': 15, 'n_estimators': 150, 'num_leaves': 100, 'subsample': 0.7}

print("===========================================================")
print("    MEJORES HIPERPARÁMETROS ENCONTRADOS Y HUMANIZADOS")
print("===========================================================")
for param, value in best_params.items():
    print(f"- {param}: {value}")
print("===========================================================")

    MEJORES HIPERPARÁMETROS ENCONTRADOS Y HUMANIZADOS
- colsample_bytree: 0.69
- learning_rate: 0.06
- max_depth: 9
- min_child_samples: 15
- n_estimators: 150
- num_leaves: 100
- subsample: 0.7


## 5. Entrenamiento del Modelo Final y Evaluación en Test

In [5]:
best_lgbm = LGBMClassifier(**best_params, random_state=42, verbose=-1)
print("Entrenando modelo final LightGBM optimizado...")
best_lgbm.fit(X_train, y_train)
print("Entrenamiento finalizado exitosamente")

y_pred_opt = best_lgbm.predict(X_test)
y_proba_opt = best_lgbm.predict_proba(X_test)[:, 1]

print("\n==================================================================")
print("          EVALUACIÓN DE RENDIMIENTO: LIGHTGBM OPTIMIZADO")
print("==================================================================")
cm_opt = confusion_matrix(y_test, y_pred_opt)
tn_opt, fp_opt, fn_opt, tp_opt = cm_opt.ravel()
print(f"Matriz de Confusión:")
print(f"   [Verdaderos Negativos (Ausencias correctas)]: {tn_opt}")
print(f"   [Falsos Positivos (Falsas alarmas)]:          {fp_opt}")
print(f"   [Falsos Negativos (Incendios no detectados)]: {fn_opt}")
print(f"   [Verdaderos Positivos (Incendios correctos)]: {tp_opt}")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred_opt, target_names=['Ausencia (0)', 'Incendio (1)']))
print(f"Área Bajo la Curva ROC (AUC-ROC): {roc_auc_score(y_test, y_proba_opt):.5f}")
print("==================================================================")

Entrenando modelo final LightGBM optimizado...
¡Entrenamiento finalizado exitosamente!

          EVALUACIÓN DE RENDIMIENTO: LIGHTGBM OPTIMIZADO
Matriz de Confusión:
   [Verdaderos Negativos (Ausencias correctas)]: 798
   [Falsos Positivos (Falsas alarmas)]:          98
   [Falsos Negativos (Incendios no detectados)]: 158
   [Verdaderos Positivos (Incendios correctos)]: 737

Reporte de Clasificación:
              precision    recall  f1-score   support

Ausencia (0)       0.83      0.89      0.86       896
Incendio (1)       0.88      0.82      0.85       895

    accuracy                           0.86      1791
   macro avg       0.86      0.86      0.86      1791
weighted avg       0.86      0.86      0.86      1791

Área Bajo la Curva ROC (AUC-ROC): 0.94002


## 6. Guardado del Modelo Final

In [6]:
import joblib
joblib.dump(best_lgbm, BASE_DIR / "lgbm_model_altB.pkl")
print("Modelo serializado con éxito en lgbm_model_altB.pkl")

Modelo serializado con éxito en lgbm_model_altB.pkl
